# Train YOLOv8 Skin Lesion Detector

Fine-tunes YOLOv8-nano on the ISIC skin lesion dataset to detect and draw bounding boxes around skin lesions.

**Runtime**: Use a GPU runtime (T4 on free Colab). Training takes ~15-30 min.

## 1. Install Dependencies

In [ ]:
!pip install ultralytics roboflow -q

## 2. Download Dataset

We use the ISIC 2018 lesion segmentation dataset from Roboflow, pre-converted to YOLO bounding box format.

**Option A**: Use Roboflow (recommended — handles format conversion automatically)

In [ ]:
from roboflow import Roboflow

# Your Roboflow API key
rf = Roboflow(api_key="any78jlnAqKX88UoTEK6")

# Download from Roboflow Universe — use your workspace slug
# Browse https://universe.roboflow.com and search "skin lesion detection"
# Then fork/clone the dataset to your workspace and use it here:
project = rf.workspace("sanikas-workspace-bopkm").project("YOUR_PROJECT_NAME")
dataset = project.version(1).download("yolov8")

DATASET_PATH = dataset.location
print(f"Dataset downloaded to: {DATASET_PATH}")

**Option B**: Build your own dataset from ISIC segmentation masks

If you have ISIC images + segmentation masks, run this cell to convert masks to YOLO bounding boxes.

In [ ]:
import os
import cv2
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
import shutil

# ---- CONFIGURE THESE PATHS ----
IMAGES_DIR = "path/to/isic/images"       # folder of .jpg images
MASKS_DIR = "path/to/isic/masks"          # folder of .png segmentation masks
OUTPUT_DIR = "lesion_dataset"              # output in YOLO format
# --------------------------------

def mask_to_yolo_bbox(mask_path, img_w, img_h):
    """Convert a binary segmentation mask to YOLO bbox format."""
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None
    _, binary = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    # Take the largest contour
    c = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(c)
    # Convert to YOLO format: class x_center y_center width height (all normalized)
    x_center = (x + w / 2) / img_w
    y_center = (y + h / 2) / img_h
    return f"0 {x_center:.6f} {y_center:.6f} {w / img_w:.6f} {h / img_h:.6f}"


# Collect image-mask pairs
images = sorted(Path(IMAGES_DIR).glob("*.jpg"))
pairs = []
for img_path in images:
    mask_name = img_path.stem + "_segmentation.png"
    mask_path = Path(MASKS_DIR) / mask_name
    if not mask_path.exists():
        # Try without _segmentation suffix
        mask_path = Path(MASKS_DIR) / (img_path.stem + ".png")
    if mask_path.exists():
        pairs.append((img_path, mask_path))

print(f"Found {len(pairs)} image-mask pairs")

# Split into train/val
train_pairs, val_pairs = train_test_split(pairs, test_size=0.2, random_state=42)

# Create YOLO directory structure
for split, split_pairs in [("train", train_pairs), ("val", val_pairs)]:
    img_dir = Path(OUTPUT_DIR) / split / "images"
    lbl_dir = Path(OUTPUT_DIR) / split / "labels"
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    for img_path, mask_path in split_pairs:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]
        label = mask_to_yolo_bbox(mask_path, w, h)
        if label is None:
            continue
        shutil.copy(img_path, img_dir / img_path.name)
        (lbl_dir / (img_path.stem + ".txt")).write_text(label)

# Create data.yaml
yaml_content = f"""path: {os.path.abspath(OUTPUT_DIR)}
train: train/images
val: val/images

names:
  0: lesion
"""
(Path(OUTPUT_DIR) / "data.yaml").write_text(yaml_content)
DATASET_PATH = OUTPUT_DIR
print(f"Dataset created at: {DATASET_PATH}")

## 3. Train YOLOv8

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8-nano (smallest, fastest)
model = YOLO("yolov8n.pt")

# Fine-tune on skin lesion data
results = model.train(
    data=f"{DATASET_PATH}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="lesion_detector",
    patience=10,        # early stopping
    device=0,           # GPU 0; use 'cpu' if no GPU
)

## 4. Evaluate

In [ ]:
# Validate on val set
metrics = model.val()
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

## 5. Test on Sample Images

In [ ]:
from IPython.display import display
from PIL import Image
import glob

# Grab a few validation images
val_images = glob.glob(f"{DATASET_PATH}/val/images/*.jpg")[:4]

for img_path in val_images:
    results = model(img_path, conf=0.25)
    annotated = results[0].plot()  # numpy array with boxes drawn
    display(Image.fromarray(annotated))
    print(f"Detections: {len(results[0].boxes)}")
    print("---")

## 6. Export Model

Copy the best weights to your SkinScreener backend.

In [ ]:
import shutil
from pathlib import Path

# Best weights are saved here after training
best_weights = Path("runs/detect/lesion_detector/weights/best.pt")
output_name = "lesion_yolo.pt"

shutil.copy(best_weights, output_name)
print(f"Model saved as: {output_name}")
print(f"Model size: {best_weights.stat().st_size / 1024 / 1024:.1f} MB")
print()
print("To use in SkinScreener:")
print(f"  Copy {output_name} to backend/model/lesion_yolo.pt")

In [ ]:
# Download the model file (for Colab)
try:
    from google.colab import files
    files.download(output_name)
except ImportError:
    print(f"Not running in Colab. Find the model at: {Path(output_name).absolute()}")